# AQNG vs PennyLane QNG on a real dataset

This notebook compares:

- **AQNG** from `AHDMarwan/aqng`
- **PennyLane QNG** with a batch-averaged block-diagonal Fubini–Study metric

on a binary subset of the real **Iris** dataset.

The comparison uses the same VQC architecture, initial parameters, mini-batches, loss, and number of steps.

AQNG uses the accessible metric $G_{\rm acc}=B^{-1}\sum_b J_b^\top\Sigma_b^+J_b$.

In [ ]:
!pip -q install "pennylane>=0.45,<0.46" scikit-learn pandas matplotlib
!pip -q install --upgrade "git+https://github.com/AHDMarwan/aqng.git"

import pennylane as qml
from pennylane import numpy as np
import numpy as onp
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from aqng_pennylane import AQNGOptimizer
print('PennyLane:', qml.__version__)

In [ ]:
SEED=7
rng=onp.random.default_rng(SEED)
iris=load_iris(); mask=iris.target<2
X=iris.data[mask].astype(float); y=iris.target[mask].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=SEED,stratify=y)
scaler=StandardScaler(); X_train=scaler.fit_transform(X_train); X_test=scaler.transform(X_test)
X_train=onp.pi*onp.tanh(X_train/2.0); X_test=onp.pi*onp.tanh(X_test/2.0)
y_train_pm=2.0*y_train-1.0; y_test_pm=2.0*y_test-1.0
print(X_train.shape,X_test.shape)

In [ ]:
N_QUBITS=4; N_LAYERS=2
dev=qml.device('default.qubit',wires=N_QUBITS)
z_terms=[(i,) for i in range(N_QUBITS)]+[(i,j) for i in range(N_QUBITS) for j in range(i+1,N_QUBITS)]
def ansatz(theta,x):
    qml.AngleEmbedding(x,wires=range(N_QUBITS),rotation='Y')
    for l in range(N_LAYERS):
        for w in range(N_QUBITS): qml.Rot(*theta[l,w],wires=w)
        for w in range(N_QUBITS): qml.CNOT(wires=[w,(w+1)%N_QUBITS])
@qml.qnode(dev,interface='autograd',diff_method='parameter-shift')
def pred_qnode(theta,x):
    ansatz(theta,x); return qml.expval(qml.PauliZ(0))
@qml.qnode(dev,interface='autograd',diff_method='parameter-shift')
def feature_qnode(theta,x):
    ansatz(theta,x); obs=[]
    for term in z_terms:
        op=qml.PauliZ(term[0])
        for w in term[1:]: op=op@qml.PauliZ(w)
        obs.append(qml.expval(op))
    return tuple(obs)
@qml.qnode(dev,interface='autograd',diff_method='parameter-shift')
def prob_qnode(theta,x):
    ansatz(theta,x); return qml.probs(wires=range(N_QUBITS))
basis=onp.arange(2**N_QUBITS); bits=((basis[:,None]>>onp.arange(N_QUBITS-1,-1,-1))&1); zvals=1.0-2.0*bits
SIGN=onp.stack([onp.prod(zvals[:,list(term)],axis=1) for term in z_terms],axis=1)
print('readout rank',len(z_terms),'params',N_LAYERS*N_QUBITS*3)

In [ ]:
def make_batch_fns(Xb,yb):
    Xb=onp.asarray(Xb); yb=onp.asarray(yb)
    def cost(theta):
        preds=qml.math.stack([pred_qnode(theta,x) for x in Xb]); return qml.math.mean((preds-yb)**2)
    def features(theta):
        return qml.math.stack([qml.math.stack(feature_qnode(theta,x)) for x in Xb])
    def covariance(theta):
        covs=[]
        for x in Xb:
            p=onp.asarray(qml.math.toarray(prob_qnode(theta,x)),dtype=float); mean=p@SIGN; second=SIGN.T@(p[:,None]*SIGN)
            covs.append(second-onp.outer(mean,mean))
        return onp.stack(covs)
    return cost,features,covariance
single_qng_metric=qml.metric_tensor(pred_qnode,approx='block-diag')
def make_qng_metric(Xb):
    Xb=onp.asarray(Xb)
    def metric(theta):
        p=int(onp.prod(theta.shape)); mats=[qml.math.reshape(single_qng_metric(theta,x),(p,p)) for x in Xb]
        return qml.math.mean(qml.math.stack(mats),axis=0)
    return metric

In [ ]:
STEPS=20; BATCH_SIZE=10; AQNG_LR=0.03; QNG_LR=0.03; LAM=1e-3; COV_LAM=1e-3
theta0=np.array(rng.normal(scale=0.15,size=(N_LAYERS,N_QUBITS,3)),requires_grad=True)
batch_ids=[rng.choice(len(X_train),size=BATCH_SIZE,replace=False) for _ in range(STEPS)]
def full_loss(theta,X,ypm):
    vals=qml.math.stack([pred_qnode(theta,x) for x in X]); return float(qml.math.mean((vals-ypm)**2))
def accuracy(theta,X,y):
    pred=onp.array([float(pred_qnode(theta,x)) for x in X]); return float(onp.mean((pred>=0).astype(int)==y))

In [ ]:
theta_a=np.array(theta0,requires_grad=True); aqng=AQNGOptimizer(stepsize=AQNG_LR,lam=LAM,cov_lam=COV_LAM,rcond=1e-8)
hist_a=[]; t0=time.perf_counter()
for step,ids in enumerate(batch_ids):
    cost,features,covariance=make_batch_fns(X_train[ids],y_train_pm[ids])
    theta_a,old=aqng.step_and_cost(cost,theta_a,feature_fn=features,covariance_fn=covariance)
    row={'optimizer':'AQNG','step':step+1,'batch_loss_before':float(old),'train_loss':full_loss(theta_a,X_train,y_train_pm),'test_loss':full_loss(theta_a,X_test,y_test_pm),'test_acc':accuracy(theta_a,X_test,y_test),'elapsed_s':time.perf_counter()-t0}
    hist_a.append(row); print('AQNG',step+1,row['test_loss'],row['test_acc'])

In [ ]:
theta_q=np.array(theta0,requires_grad=True); qng=qml.QNGOptimizer(stepsize=QNG_LR,approx=None,lam=LAM)
hist_q=[]; t0=time.perf_counter()
for step,ids in enumerate(batch_ids):
    cost,_,_=make_batch_fns(X_train[ids],y_train_pm[ids]); metric_fn=make_qng_metric(X_train[ids])
    theta_q,old=qng.step_and_cost(cost,theta_q,metric_tensor_fn=metric_fn)
    row={'optimizer':'PennyLane-QNG','step':step+1,'batch_loss_before':float(old),'train_loss':full_loss(theta_q,X_train,y_train_pm),'test_loss':full_loss(theta_q,X_test,y_test_pm),'test_acc':accuracy(theta_q,X_test,y_test),'elapsed_s':time.perf_counter()-t0}
    hist_q.append(row); print('QNG',step+1,row['test_loss'],row['test_acc'])

In [ ]:
df=pd.DataFrame(hist_a+hist_q); display(df.groupby('optimizer').tail(1))
plt.figure(figsize=(7,4))
for name,g in df.groupby('optimizer'): plt.plot(g.step,g.test_loss,marker='o',label=name)
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('test MSE'); plt.legend(); plt.grid(alpha=.25); plt.show()
plt.figure(figsize=(7,4))
for name,g in df.groupby('optimizer'): plt.plot(g.step,g.test_acc,marker='o',label=name)
plt.xlabel('step'); plt.ylabel('test accuracy'); plt.ylim(0,1.02); plt.legend(); plt.grid(alpha=.25); plt.show()
df.to_csv('/content/aqng_vs_qng_history.csv',index=False)